In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from typing import Dict, List, Tuple
from data_loading import *
from loss_funcs import *

print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}") # I have a 3090
    torch.set_float32_matmul_precision("medium")
    print("float32 matmul precision set to 'medium'")

# ── Moirai / training-stack imports ──────────────────────────────────────────
# pip install "uni2ts @ git+https://github.com/SalesforceAIResearch/uni2ts.git" gluonts mlflow
# Requires Python >= 3.10 (uni2ts 2.x)
import os
import copy
from tqdm.notebook import tqdm

import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from gluonts.dataset.common import ListDataset

# uni2ts 2.x: MoiraiModule owns from_pretrained; MoiraiForecast/MoiraiFinetune take module=
from uni2ts.model.moirai import MoiraiForecast, MoiraiFinetune
from uni2ts.model.moirai.module import MoiraiModule

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, TQDMProgressBar

import mlflow
import mlflow.pytorch
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device           : {DEVICE}")


In [12]:
weather_cols_all = ['temperature_2m',
       'apparent_temperature', 'dew_point_2m', 'relative_humidity_2m',
       'precipitation', 'rain', 'snowfall', 'cloud_cover', 'cloud_cover_low',
       'cloud_cover_mid', 'cloud_cover_high', 'surface_pressure',
       'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m',
       'shortwave_radiation', 'diffuse_radiation', 'direct_normal_irradiance']

other_cols = [ # these are not static
    'dam_price', 'buy_bm_price', 'sell_bm_price',
    'max_power', 'max_solar', 'max_ev'
]

cat_columns = [
    'eic_code', 'dso_desc', 'station_type', 'oblast',
    'Month', 'Day', 'Hour', 'day_of_week', 'season'
]

static_cols = [
    'latitude', 'longitude', 'eic_code', 'dso_desc', 'station_type', 'oblast'
]

calendar_cols = [
    'Month', 'Day', 'Hour', 'day_of_week', 'season'
]

time_cols = ['datetime', 'time_idx']

FUTURE_REALS = weather_cols_all + calendar_cols + static_cols + other_cols
print(f"y col is: {Y_COL}, group col is: {GROUP_COL}\n"
      f"features: {FUTURE_REALS}")

y col is: sum_of_kWh, group col is: eic_code
features: ['temperature_2m', 'apparent_temperature', 'dew_point_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'snowfall', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high', 'surface_pressure', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m', 'shortwave_radiation', 'diffuse_radiation', 'direct_normal_irradiance', 'Month', 'Day', 'Hour', 'day_of_week', 'season', 'latitude', 'longitude', 'eic_code', 'dso_desc', 'station_type', 'oblast', 'dam_price', 'buy_bm_price', 'sell_bm_price', 'max_power', 'max_solar', 'max_ev']


In [4]:
# this data has a data column and categorical columns are kept intact and will need to be handled.
# "time_idx" is already built in train, val and test and is continuous through them
print("Loading train …")
train = load_and_prepare(TRAIN_PATH_WITH_DATETME)

print("Loading val   …")
val = load_and_prepare(VAL_PATH_WITH_DATETME)

print("Loading test  …")
test = load_and_prepare(TEST_PATH_WITH_DATETME)

print(f"train: {train.shape}")
print(f"val : {val.shape}")
print(f"test: {test.shape}")

Loading train …
Loading val   …
Loading test  …
train: (4718010, 38)
val : (293880, 38)
test: (295430, 38)


In [ ]:
# # If a model cant handle categorical columns natively, or through embedings use this data
# # It has no datetime column and all columns are numeric, as all cat column were ohe
# # GROUP_COL is the only exception, and is not ohe. Ohe it before training
# print("Loading train …")
# train = load_and_prepare(TRAIN_PATH_OHE)
#
# print("Loading val   …")
# val = load_and_prepare(VAL_PATH_OHE)
#
# print("Loading test  …")
# test = load_and_prepare(TEST_PATH_OHE)
#
# print(f"train: {train.shape}")
# print(f"val  : {val.shape}")
# print(f"test : {test.shape}")

In [ ]:
# this cell samples locations, I'll use it if training takes too long, otherwise don't touch it

TARGET_STATIONS = 395

station_stats = (
    train.groupby(GROUP_COL)
    .agg(rows=(Y_COL, "count"))
    .reset_index()
    .sort_values("rows", ascending=False)
)

sampled_stations = station_stats.sample(
    n=TARGET_STATIONS, random_state=42
)[GROUP_COL].values

print(f"Stations: {len(sampled_stations)}")

train = train[train[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
val   = val[val[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
test  = test[test[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)

print(f"Train rows : {len(train):,}")
print(f"Val rows   : {len(val):,}")
print(f"Test rows  : {len(test):,}")

In [ ]:
training_cutoff = train["time_idx"].max()
val_cutoff      = val["time_idx"].max()
test_cutoff     = test["time_idx"].max()

print(f"training cutoff : {training_cutoff}")
print(f"val cutoff      : {val_cutoff}")
print(f"test cutoff     : {test_cutoff}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Moirai config — sizing and training hyperparameters
# ─────────────────────────────────────────────────────────────────────────────
# Moirai is a *foundation* time-series model (Salesforce, ICML 2024). It works
# in patch space: each window of length CONTEXT_LEN + PRED_LEN is reshaped into
# (n_patches, patch_size). PATCH_SIZE must divide CONTEXT_LEN + PRED_LEN.

MODEL_ID    = "Salesforce/moirai-1.1-R-small"   # small=91M / base=236M / large=311M
CONTEXT_LEN = 168    # 1-week look-back (hourly)
PRED_LEN    = 48     # 48-hour forecast horizon (matches val/test rolling)
PATCH_SIZE  = 8      # (168 + 48) / 8 = 27 patches; last 6 are prediction patches
NUM_SAMPLES = 100    # Monte-Carlo samples for the predictive distribution

# Fine-tuning
TRAIN_STRIDE = 24    # one window per day — fast, diverse coverage
BATCH_SIZE   = 64
EPOCHS       = 10
LR           = 1e-4
WEIGHT_DECAY = 1e-2
NUM_WORKERS  = 0     # 0 on Windows / Jupyter (avoids multiprocessing deadlock)
SEED         = 42

# Money-loss training mode:
#   "nll"    — SAFE PATH (default). Fine-tunes with Moirai's native PackedNLLLoss
#              (the loss the source notebook uses, validated to work). Money_pct
#              is computed only at evaluation. Set this first to confirm the
#              full pipeline runs end-to-end on your installed uni2ts version.
#   "custom" — EXPERIMENTAL. Subclasses MoiraiFinetune and overrides
#              training_step to optimize money_pct on the predictive
#              distribution's mean. The exact MoiraiModule.forward signature is
#              version-dependent in uni2ts and can change between releases (see
#              uni2ts discussion #57 — even Salesforce's own PackedMSELoss had
#              shape bugs against the Distribution output). Validate against
#              your installed version before relying on results.
TRAINING_MODE = "nll"   # "nll" (default, safe) | "custom" (money_pct override)

CKPT_DIR  = "../checkpoints"
SAVE_PATH = os.path.join(CKPT_DIR, "moirai_finetuned_money.pt")
os.makedirs(CKPT_DIR, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)


## Moirai dataset preparation

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Moirai-specific dataset preparation
# ─────────────────────────────────────────────────────────────────────────────
# Moirai (uni2ts v2.x) consumes data in PATCH form, not raw (T, 1) tensors.
# Each sample is a dict whose tensors all have a leading patch dim of size P:
#   target          : (P, MAX_PATCH=128) float  — values, zero-padded after patch_size
#   observed_mask   : (P, MAX_PATCH)     bool   — True where target is real data
#   prediction_mask : (P,)               bool   — True for the last n_pred_patches
#   time_id         : (P,)               long   — patch position 0..P-1
#   variate_id      : (P,)               long   — variate index (0 for univariate)
#   patch_size      : (P,)               long   — patch size used for this sample
#   sample_id       : (P,)               long   — segment id (single segment → 0)
#
# This Moirai example uses a UNIVARIATE target (Y_COL only). Exogenous reals
# *can* be passed via past_feat_dynamic_real / feat_dynamic_real, but the
# dataset class shape changes substantially and the working uni2ts template
# does not use them — so we keep target-only here. Weather, calendar, and
# price columns still feed the **money loss** through the eval merge, and the
# foundation model already encodes calendar effects implicitly.
# Categorical columns are NOT used as model inputs in this univariate setup;
# eic_code is used only to split per-series windows.

def build_series_dict(df: pd.DataFrame) -> Dict[str, dict]:
    out = {}
    for eic, grp in df.groupby(GROUP_COL):
        grp = grp.sort_values("time_idx")
        out[eic] = {
            "values":   grp[Y_COL].values.astype(np.float32),
            "time_idx": grp["time_idx"].values.astype(np.int64),
            "datetime": grp["datetime"].values,
        }
    return out


def fill_nan(arr: np.ndarray) -> np.ndarray:
    if not np.isnan(arr).any():
        return arr
    return (
        pd.Series(arr)
        .interpolate(method="linear", limit_direction="both")
        .values.astype(np.float32)
    )


# Continuous full-history view for rolling inference (val + test windows need
# train-side context, so we concat). time_idx is already continuous across
# splits per the project setup.
full_df = (
    pd.concat([train, val, test], ignore_index=True)
    .sort_values([GROUP_COL, "time_idx"])
    .reset_index(drop=True)
)

train_series = build_series_dict(train)
full_series  = build_series_dict(full_df)

lengths = [len(v["values"]) for v in train_series.values()]
print(f"Stations         : {len(train_series)}")
print(f"Train series len : min={min(lengths)}, max={max(lengths)}, mean={np.mean(lengths):.0f}")


class MoiraiWindowDataset(Dataset):
    """Sliding-window dataset producing the patch-shaped tensors MoiraiFinetune expects."""

    MAX_PATCH = 128  # moirai-1.1-R fixed max patch size — pad shorter patches to this width

    def __init__(self, series_dict, context_len, pred_len, stride, patch_size):
        self.context_len    = context_len
        self.pred_len       = pred_len
        self.seq_len        = context_len + pred_len
        self.patch_size     = patch_size
        self.n_patches      = self.seq_len // patch_size
        self.n_pred_patches = pred_len // patch_size

        self.windows: List[np.ndarray] = []
        for eic, data in series_dict.items():
            ts = fill_nan(data["values"])
            if len(ts) < self.seq_len:
                continue
            for s in range(0, len(ts) - self.seq_len + 1, stride):
                self.windows.append(ts[s : s + self.seq_len].copy())

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        seq = self.windows[idx]
        P, ps = self.n_patches, self.patch_size

        # Per-instance normalisation using context-window statistics only.
        ctx = seq[: self.context_len]
        mu  = ctx.mean()
        sig = max(ctx.std(), 1e-5)
        seq_n = (seq - mu) / sig

        target = torch.zeros(P, self.MAX_PATCH, dtype=torch.float32)
        target[:, :ps] = torch.from_numpy(seq_n.reshape(P, ps)).float()

        observed_mask = torch.zeros(P, self.MAX_PATCH, dtype=torch.bool)
        observed_mask[:, :ps] = True

        prediction_mask = torch.zeros(P, dtype=torch.bool)
        prediction_mask[-self.n_pred_patches :] = True

        return {
            "target":          target,
            "observed_mask":   observed_mask,
            "prediction_mask": prediction_mask,
            "time_id":         torch.arange(P, dtype=torch.long),
            "variate_id":      torch.zeros(P, dtype=torch.long),
            "patch_size":      torch.full((P,), ps, dtype=torch.long),
            "sample_id":       torch.zeros(P, dtype=torch.long),
        }


def collate_moirai(batch):
    return {k: torch.stack([b[k] for b in batch]) for k in batch[0]}


train_ds = MoiraiWindowDataset(train_series, CONTEXT_LEN, PRED_LEN, TRAIN_STRIDE, PATCH_SIZE)
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_moirai, num_workers=NUM_WORKERS, pin_memory=True,
)
print(f"Training windows : {len(train_ds):,}")
print(f"Batches / epoch  : {len(train_loader):,}")
print(f"Patches / sample : {train_ds.n_patches}  (pred patches: {train_ds.n_pred_patches})")


## Money-loss wrapper (differentiable PyTorch)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Differentiable PyTorch wrapper around money_pct
# ─────────────────────────────────────────────────────────────────────────────
# loss_funcs.money_pct works on numpy arrays for evaluation. For TRAINING we
# need a vectorized, differentiable PyTorch version that follows the same
# Ukrainian intraday-market rule:
#   • |error| within ±10 % of actual → no penalty (only DAM cost on the deviation)
#   • over-prediction beyond +10 %  → buy back excess at sell_bm_price (≈ −40 % discount)
#   • under-prediction beyond −10 % → cover deficit at buy_bm_price (≈ +40 % surcharge)
# Money_pct expresses the resulting cost as a percentage of the perfect-foresight DAM cost.
# All three price tensors must have the same shape as y_true / y_pred.

def money_pct_torch(
    y_true: torch.Tensor,     # (B,) — actual kWh
    y_pred: torch.Tensor,     # (B,) — predicted kWh
    dam_price: torch.Tensor,  # (B,) — day-ahead price
    sell_bm_price: torch.Tensor,  # (B,) — balancing sell-back price (≤ dam)
    buy_bm_price: torch.Tensor,   # (B,) — balancing buy price (≥ dam)
    tol: float = 0.10,
    eps: float = 1e-6,
) -> torch.Tensor:
    error = y_pred - y_true
    abs_tol = tol * torch.clamp(y_true.abs(), min=eps)

    # Within tolerance → just pay DAM for what was actually consumed
    inside = error.abs() <= abs_tol

    # Over-prediction outside tolerance: ordered too much → sell back excess
    over_excess = torch.clamp(error - abs_tol, min=0.0)
    # Under-prediction outside tolerance: ordered too little → buy deficit
    under_deficit = torch.clamp(-error - abs_tol, min=0.0)

    # Cost components (DAM is paid for what was *ordered* + balancing adjustment)
    cost_perfect = y_true * dam_price
    cost_dam     = y_pred * dam_price
    cost_buy     = under_deficit * buy_bm_price       # extra spend
    cost_sell    = over_excess  * sell_bm_price       # rebate (subtracted)

    cost_actual  = torch.where(
        inside,
        cost_dam,                                     # no balancing market touched
        cost_dam + cost_buy - cost_sell,
    )
    pct = (cost_actual - cost_perfect) / torch.clamp(cost_perfect.abs(), min=eps) * 100.0
    return pct.mean()


# ─────────────────────────────────────────────────────────────────────────────
# Custom MoiraiFinetune that swaps NLL for money_pct on the predicted point
# forecast (mean of the predictive distribution).
# ─────────────────────────────────────────────────────────────────────────────
# Why subclass: MoiraiFinetune.training_step computes PackedNLLLoss internally
# from a mixture distribution. There is no constructor switch for a custom
# point-forecast loss. The clean integration point is overriding training_step:
# we still run the same forward pass to obtain the predictive distribution,
# but compute loss as money_pct(distribution.mean, target) instead of NLL.
#
# Trade-offs to be aware of:
#   • We lose the probabilistic training signal (sharpness of the distribution).
#   • Money_pct is *not* convex; gradients can be sparse where the error is
#     inside the ±10 % band. We add a tiny MAE regulariser to keep gradients
#     informative everywhere.
#   • Prices are not in the Moirai input pipeline (univariate target). For the
#     loss we need per-window price tensors; we hold a price lookup keyed by
#     (eic, time_idx) and join in training_step. For simplicity here, since
#     the dataset above is order-preserving but does not carry eic/time_idx
#     through, we apply a *price-agnostic* approximation: use a global mean
#     price triplet for training (computed below). For the final reported
#     evaluation we still compute money_pct with TRUE per-hour prices.
#
# This is honest about the limitation: training sees a uniform-price money
# loss, evaluation sees the real per-hour-price money loss. If you want
# per-window real prices in training, extend MoiraiWindowDataset to carry
# the three price arrays alongside the target window.

# Global average prices (used inside the training loss as a price-agnostic proxy)
TRAIN_DAM_MEAN  = float(train["dam_price"].mean())
TRAIN_SELL_MEAN = float(train["sell_bm_price"].mean())
TRAIN_BUY_MEAN  = float(train["buy_bm_price"].mean())
print(f"Mean prices used for money-loss training:")
print(f"  DAM   : {TRAIN_DAM_MEAN:.4f}")
print(f"  SELL  : {TRAIN_SELL_MEAN:.4f}")
print(f"  BUY   : {TRAIN_BUY_MEAN:.4f}")


class MoneyLossMoiraiFinetune(MoiraiFinetune):
    """Replaces PackedNLLLoss with money_pct on the distribution mean."""

    def __init__(self, *args, mae_weight: float = 0.1,
                 dam_mean: float = 1.0, sell_mean: float = 0.6, buy_mean: float = 1.4,
                 **kwargs):
        super().__init__(*args, **kwargs)
        self.mae_weight = mae_weight
        self.register_buffer("dam_mean",  torch.tensor(dam_mean,  dtype=torch.float32))
        self.register_buffer("sell_mean", torch.tensor(sell_mean, dtype=torch.float32))
        self.register_buffer("buy_mean",  torch.tensor(buy_mean,  dtype=torch.float32))

    def training_step(self, batch, batch_idx):
        # Forward pass — Moirai returns a torch.distributions.Distribution
        distr = self.module(
            target          = batch["target"],
            observed_mask   = batch["observed_mask"],
            sample_id       = batch["sample_id"],
            time_id         = batch["time_id"],
            variate_id      = batch["variate_id"],
            prediction_mask = batch["prediction_mask"],
            patch_size      = batch["patch_size"],
        )

        # Mean of the predictive distribution → point forecast
        # Shape: (B, P, max_patch). Select prediction patches only.
        point  = distr.mean
        target = batch["target"]
        pmask  = batch["prediction_mask"]      # (B, P)
        omask  = batch["observed_mask"]        # (B, P, max_patch)

        # Combined mask: prediction patches AND observed positions inside them
        combined = pmask.unsqueeze(-1) & omask
        y_pred = point[combined]
        y_true = target[combined]

        # Broadcast mean prices to every prediction step (price-agnostic proxy)
        dam  = self.dam_mean.expand_as(y_true)
        sell = self.sell_mean.expand_as(y_true)
        buy  = self.buy_mean.expand_as(y_true)

        money_loss = money_pct_torch(y_true, y_pred, dam, sell, buy)
        mae        = F.l1_loss(y_pred, y_true)
        loss       = money_loss + self.mae_weight * mae

        self.log("train/money_pct", money_loss, prog_bar=True, on_step=True, on_epoch=True)
        self.log("train/mae",       mae,        prog_bar=False, on_step=True, on_epoch=True)
        self.log("train/loss",      loss,       prog_bar=True, on_step=True, on_epoch=True)
        return loss


## Moirai model initialization

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Moirai model initialization
# ─────────────────────────────────────────────────────────────────────────────
# Two objects:
#   • MoiraiModule      — the raw transformer weights (loaded from HuggingFace).
#   • MoiraiForecast    — the inference wrapper that produces a GluonTS predictor.
#   • MoiraiFinetune    — the Lightning training wrapper.
# We deepcopy the module into MoiraiFinetune so the zero-shot weights stay
# clean for diagnostic comparisons.

module = MoiraiModule.from_pretrained(MODEL_ID)

zs_model = MoiraiForecast(
    module=module,
    prediction_length=PRED_LEN,
    target_dim=1,
    feat_dynamic_real_dim=0,           # univariate target
    past_feat_dynamic_real_dim=0,
    context_length=CONTEXT_LEN,
    patch_size=PATCH_SIZE,
    num_samples=NUM_SAMPLES,
)
zs_predictor = zs_model.create_predictor(batch_size=BATCH_SIZE * 2, device=DEVICE)

n_params = sum(p.numel() for p in zs_model.parameters())
print(f"Loaded   : {MODEL_ID}")
print(f"Params   : {n_params:,}  ({n_params / 1e6:.1f} M)")

num_training_steps = len(train_loader) * EPOCHS
num_warmup_steps   = min(500, num_training_steps // 10)

ft_kwargs = dict(
    module=copy.deepcopy(module),
    min_patches=4,
    min_mask_ratio=0.15,
    max_mask_ratio=0.5,
    max_dim=1,
    num_training_steps=num_training_steps,
    num_warmup_steps=num_warmup_steps,
    context_length=CONTEXT_LEN,
    prediction_length=PRED_LEN,
    patch_size=PATCH_SIZE,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

if TRAINING_MODE == "custom":
    ft_model = MoneyLossMoiraiFinetune(
        **ft_kwargs,
        mae_weight=0.1,
        dam_mean=TRAIN_DAM_MEAN,
        sell_mean=TRAIN_SELL_MEAN,
        buy_mean=TRAIN_BUY_MEAN,
    )
    monitor_metric = "train/money_pct"
    print("Training objective: money_pct (custom) + 0.1 · MAE  [EXPERIMENTAL]")
elif TRAINING_MODE == "nll":
    ft_model = MoiraiFinetune(**ft_kwargs)
    monitor_metric = "train/PackedNLLLoss"
    print("Training objective: PackedNLLLoss (Moirai native, safe default)")
else:
    raise ValueError(f"TRAINING_MODE must be 'nll' or 'custom', got: {TRAINING_MODE!r}")

print(f"Trainable params: {sum(p.numel() for p in ft_model.parameters() if p.requires_grad):,}")
print(f"Training steps  : {num_training_steps}  |  warmup: {num_warmup_steps}")


## Training loop + MLflow logging

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Training — Lightning Trainer + MLflow run
# ─────────────────────────────────────────────────────────────────────────────
mlflow.set_experiment("moirai_electricity_forecasting")
mlflow.start_run(run_name=f"moirai-{TRAINING_MODE}-ctx{CONTEXT_LEN}-pred{PRED_LEN}")

# ─── Log run-level config / metadata ─────────────────────────────────────────
mlflow.log_params({
    "model_id":         MODEL_ID,
    "context_len":      CONTEXT_LEN,
    "pred_len":         PRED_LEN,
    "patch_size":       PATCH_SIZE,
    "num_samples":      NUM_SAMPLES,
    "train_stride":     TRAIN_STRIDE,
    "batch_size":       BATCH_SIZE,
    "epochs":           EPOCHS,
    "lr":               LR,
    "weight_decay":     WEIGHT_DECAY,
    "loss":             "money_pct+0.1*mae" if TRAINING_MODE == "custom" else "PackedNLLLoss",
    "training_mode":    TRAINING_MODE,
    "y_col":            Y_COL,
    "group_col":        GROUP_COL,
    "n_stations":       len(train_series),
    "train_rows":       len(train),
    "val_rows":         len(val),
    "test_rows":        len(test),
    "train_windows":    len(train_ds),
    "train_date_min":   str(train["datetime"].min()),
    "train_date_max":   str(train["datetime"].max()),
    "val_date_min":     str(val["datetime"].min()),
    "val_date_max":     str(val["datetime"].max()),
    "test_date_min":    str(test["datetime"].min()),
    "test_date_max":    str(test["datetime"].max()),
})
mlflow.log_dict({"future_reals": FUTURE_REALS, "cat_columns": cat_columns}, "feature_columns.json")

# ─── Lightning callbacks ─────────────────────────────────────────────────────
checkpoint_cb = ModelCheckpoint(
    dirpath=CKPT_DIR,
    filename=f"moirai-{{epoch:02d}}-{{{monitor_metric.replace('/', '_')}:.4f}}",
    monitor=monitor_metric,
    mode="min",
    save_top_k=1,
)

trainer = L.Trainer(
    max_epochs=EPOCHS,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    precision="16-mixed" if torch.cuda.is_available() else "32",
    callbacks=[checkpoint_cb, TQDMProgressBar(refresh_rate=10)],
    enable_progress_bar=True,
    log_every_n_steps=10,
)

trainer.fit(ft_model, train_loader)

print(f"\nBest checkpoint : {checkpoint_cb.best_model_path}")
print(f"Best score      : {checkpoint_cb.best_model_score:.4f}")

# Save fine-tuned weights and log as MLflow artifact
torch.save(ft_model.module.state_dict(), SAVE_PATH)
mlflow.log_artifact(SAVE_PATH, artifact_path="model")
if checkpoint_cb.best_model_path:
    mlflow.log_artifact(checkpoint_cb.best_model_path, artifact_path="checkpoint")
print(f"Fine-tuned weights saved → {SAVE_PATH}")

# Hot-swap weights into the inference predictor
zs_model.module.load_state_dict(ft_model.module.state_dict())
ft_predictor = zs_model.create_predictor(batch_size=BATCH_SIZE * 2, device=DEVICE)
print("Fine-tuned predictor ready")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Rolling inference — tile val / test in non-overlapping PRED_LEN blocks.
# Each block is predicted from the preceding CONTEXT_LEN hours of GROUND TRUTH.
# This matches the operational setup: every PRED_LEN hours we re-run with the
# latest realised data. (Pure recursive prediction would compound error.)
# ─────────────────────────────────────────────────────────────────────────────
def rolling_forecast(predictor, start_cutoff, end_cutoff, split_name="split", infer_batch=256):
    all_windows = []
    for eic in tqdm(full_series, desc=f"Collecting {split_name} windows"):
        data     = full_series[eic]
        values   = fill_nan(data["values"])
        tidx     = data["time_idx"]
        dts      = data["datetime"]
        tidx_map = {int(t): i for i, t in enumerate(tidx)}

        t = start_cutoff + 1
        while t <= end_cutoff:
            ctx_start = t - CONTEXT_LEN
            if ctx_start not in tidx_map or t not in tidx_map:
                t += PRED_LEN
                continue
            s, e = tidx_map[ctx_start], tidx_map[t]
            if e - s != CONTEXT_LEN:
                t += PRED_LEN
                continue

            all_windows.append({
                "eic":      eic,
                "t":        t,
                "ctx":      values[s:e],
                "ts":       values,
                "tidx_map": tidx_map,
                "start":    pd.Period(pd.Timestamp(dts[s]), freq="h"),
            })
            t += PRED_LEN

    records = []
    for b0 in tqdm(range(0, len(all_windows), infer_batch), desc=f"Predicting {split_name}"):
        batch = all_windows[b0 : b0 + infer_batch]
        ds = ListDataset(
            [{"target": w["ctx"], "start": w["start"], "item_id": w["eic"]} for w in batch],
            freq="h",
        )
        forecasts = list(predictor.predict(ds))

        for w, fc in zip(batch, forecasts):
            pred = fc.mean   # (PRED_LEN,) point forecast
            for step in range(PRED_LEN):
                ti = w["t"] + step
                if ti not in w["tidx_map"]:
                    break
                pos = w["tidx_map"][ti]
                records.append({
                    GROUP_COL:  w["eic"],
                    "time_idx": int(ti),
                    Y_COL:      float(w["ts"][pos]),
                    "pred":     float(pred[step]),
                })

    return pd.DataFrame(records)


# Run rolling inference on val and test windows
print("── Validation rolling inference ────────────────────────────")
val_pred  = rolling_forecast(ft_predictor, training_cutoff, val_cutoff, split_name="val")
print(f"  val prediction rows : {len(val_pred):,}")

print("── Test rolling inference ──────────────────────────────────")
test_pred = rolling_forecast(ft_predictor, val_cutoff, test_cutoff, split_name="test")
print(f"  test prediction rows: {len(test_pred):,}")

# Re-attach prices and datetime so loss_funcs.money / money_pct work and so the
# existing per-station + plot cells (which expect 'datetime') keep working.
PRICE_COLS = ["dam_price", "sell_bm_price", "buy_bm_price"]
keep = [GROUP_COL, "time_idx", "datetime"] + PRICE_COLS

val_eval  = val_pred.merge(val[keep],  on=[GROUP_COL, "time_idx"], how="inner")
test_eval = test_pred.merge(test[keep], on=[GROUP_COL, "time_idx"], how="inner")

# Sanity: drop any zero-actual rows that would break MAPE/money_pct denominators
val_eval  = val_eval[val_eval[Y_COL].abs() > 1e-6].reset_index(drop=True)
test_eval = test_eval[test_eval[Y_COL].abs() > 1e-6].reset_index(drop=True)

print(f"val_eval  : {val_eval.shape}")
print(f"test_eval : {test_eval.shape}")

# Bias and MAE — added so the existing rolling_eval cell can simply log them.
def _bias(df):
    return float(df["pred"].sum() - df[Y_COL].sum())

def _mae(df):
    return float((df[Y_COL] - df["pred"]).abs().mean())

val_bias_v,  test_bias_v = _bias(val_eval),  _bias(test_eval)
val_mae_v,   test_mae_v  = _mae(val_eval),   _mae(test_eval)
print(f"VAL  bias={val_bias_v:+.2f}  MAE={val_mae_v:.4f}")
print(f"TEST bias={test_bias_v:+.2f}  MAE={test_mae_v:.4f}")

mlflow.log_metrics({
    "val_bias":   val_bias_v,
    "val_mae":    val_mae_v,
    "test_bias":  test_bias_v,
    "test_mae":   test_mae_v,
})

# Persist a small sample of predictions as an MLflow artifact for inspection
sample_path = os.path.join(CKPT_DIR, "test_predictions_sample.csv")
test_eval.head(5000).to_csv(sample_path, index=False)
mlflow.log_artifact(sample_path, artifact_path="predictions")


## Validation / test metrics + MLflow logging

In [ ]:
def _prices(df):
    return df["dam_price"].values, df["sell_bm_price"].values, df["buy_bm_price"].values

val_smape_v     = smape(val_eval[Y_COL], val_eval['pred'])
val_rmse_v      = rmse(val_eval[Y_COL], val_eval['pred'])
val_mape_v      = mape(val_eval[Y_COL], val_eval['pred'])
val_money_v     = money(val_eval[Y_COL], val_eval['pred'], *_prices(val_eval))
val_money_pct_v = money_pct(val_eval[Y_COL], val_eval['pred'], *_prices(val_eval))

test_smape_v     = smape(test_eval[Y_COL], test_eval['pred'])
test_rmse_v      = rmse(test_eval[Y_COL], test_eval['pred'])
test_mape_v      = mape(test_eval[Y_COL], test_eval['pred'])
test_money_v     = money(test_eval[Y_COL], test_eval['pred'], *_prices(test_eval))
test_money_pct_v = money_pct(test_eval[Y_COL], test_eval['pred'], *_prices(test_eval))

print("── Validation ──────────────────────────────────────────────")
print(f"Aligned samples : {len(val_eval):,}")
print(f"SMAPE     : {val_smape_v:.4f}")
print(f"RMSE      : {val_rmse_v:.4f}")
print(f"MAPE      : {val_mape_v:.2f} %")
print(f"MONEY     : {val_money_v:.4f}")
print(f"MONEY_PCT : {val_money_pct_v:.4f}%")

print("── Test ────────────────────────────────────────────────────")
print(f"Aligned samples : {len(test_eval):,}")
print(f"SMAPE     : {test_smape_v:.4f}")
print(f"RMSE      : {test_rmse_v:.4f}")
print(f"MAPE      : {test_mape_v:.2f} %")
print(f"MONEY     : {test_money_v:.4f}")
print(f"MONEY_PCT : {test_money_pct_v:.4f}%")

mlflow.log_metrics({
    "val_smape":      val_smape_v,
    "val_rmse":       val_rmse_v,
    "val_mape":       val_mape_v,
    "val_money":      val_money_v,
    "val_money_pct":  val_money_pct_v,
    "test_smape":     test_smape_v,
    "test_rmse":      test_rmse_v,
    "test_mape":      test_mape_v,
    "test_money":     test_money_v,
    "test_money_pct": test_money_pct_v,
})
mlflow.end_run()
print(f"MLflow run logged → {mlflow.get_tracking_uri()}")

In [ ]:
def per_station_metrics(eval_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for grp, gdf in eval_df.groupby(GROUP_COL):
        rows.append({
            GROUP_COL:    grp,
            "n":          len(gdf),
            "SMAPE":      smape(gdf[Y_COL], gdf["pred"]),
            "RMSE":       rmse (gdf[Y_COL], gdf["pred"]),
            "MAPE":       mape (gdf[Y_COL], gdf["pred"]),
            "MONEY":      mape (gdf[Y_COL], gdf["pred"]),
            "MONEY_PCT":  mape (gdf[Y_COL], gdf["pred"]),
        })
    return pd.DataFrame(rows).sort_values("SMAPE")


test_station_metrics = per_station_metrics(test_eval)

print("Top-10 best stations (test SMAPE):")
print(test_station_metrics.head(10).to_string(index=False))
print("\nBottom-10 worst stations (test SMAPE):")
print(test_station_metrics.tail(10).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


def plot_forecast(df, eic_code, start_dt=None, end_dt=None, title_prefix=""):

    df = df[df[GROUP_COL] == eic_code].sort_values("datetime")
    if df.empty:
        raise ValueError(f"No data for EiC code: {eic_code!r}")
    if start_dt is not None:
        df = df[df["datetime"] >= pd.Timestamp(start_dt)]
    if end_dt is not None:
        df = df[df["datetime"] <= pd.Timestamp(end_dt)]
    if df.empty:
        raise ValueError("No data in the specified datetime range.")

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(df["datetime"], df[Y_COL],  label="True",      linewidth=1, color="steelblue")
    ax.plot(df["datetime"], df["pred"], label="Predicted", linewidth=1, color="tomato", alpha=0.85)
    ax.set_title(
        f"{title_prefix}{eic_code}  |  MAPE={mape(df[Y_COL], df['pred']):.3f}"
        f"  MONEY_PCT={money_pct(df[Y_COL], df['pred'], *_prices(df)):.2f}"
        f"  ({df['datetime'].min().date()} \u2013 {df['datetime'].max().date()})"
    )
    ax.set_xlabel("Datetime")
    ax.set_ylabel(Y_COL)
    ax.legend()
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    fig.autofmt_xdate(rotation=0, ha="center")
    plt.tight_layout()
    plt.show()


# Example:
# plot_forecast(val_eval,  eic_code="<code>")
# plot_forecast(test_eval, eic_code="<code>", start_dt="2025-08-25", end_dt="2025-08-26")

In [ ]:
best_station  = test_station_metrics.iloc[0][GROUP_COL]
worst_station = test_station_metrics.iloc[-1][GROUP_COL]

print(f"Best  station (SMAPE): {best_station}")
plot_forecast(test_eval, eic_code=best_station,  title_prefix="[BEST]  ")

print(f"Worst station (SMAPE): {worst_station}")
plot_forecast(test_eval, eic_code=worst_station, title_prefix="[WORST] ")